In [1]:
""" YouTrack to Jira export script
Script pulls data from YouTrack API and converts it to a Jira CSV import compatible csv. 
"""
import os
import csv
import requests
import pandas as pd
import unpackers
from colorama import Fore, Style

In [2]:
# configuration
project_name = 'ATAT'
api_url = "https://pandatrack.myjetbrains.com/youtrack/api/"
num_issues_to_skip = 599
num_issues_to_retrieve = 10

# load api token
with open('.youtracktoken') as file:
    auth_token = file.readline().strip()

# set request headers
auth_header = {"Authorization": f"Bearer {auth_token}"}

In [3]:
# get project id based on project name
print("Getting YouTrack Project Id...")
projects_list_fields= {"fields": "id,name"}
projects_list = requests.get(api_url+"admin/projects", params=projects_list_fields, headers=auth_header)
project_id = None
for project in projects_list.json():
    if project_name in project['name']:
        project_id = project['id']

if project_id == None:
    raise ValueError(f"Project name {project_name} does not exist")

Getting YouTrack Project Id...


In [4]:
# get list of issues for the project
issues_list_fields={
    "fields": (
        "idReadable,"
        "summary,"
        "description,"
        "created,"
        "updated,"
        "reporter("
            "fullName,"
            "email,"
            "banned"
            "),"
        "updater("
            "fullName,"
            "email,"
            "banned"
            "),"
        "resolved,"
        "numberInProject,"
        "comments("
            "text,"
            "author("
                "email"
                "),"
            "created"
            "),"
        "tags("
            "name"
            "),"
        "links("
            "direction,"
            "linkType("
                "directed,"
                "name,"
                "sourceToTarget,"
                "targetToSource"
                "),"
            "issues("
                "id,"
                "idReadable"
                ")"
            "),"
        "customFields("
            "name,"
            "value("
                "name,"
                "text,"
                "minutes,"
                "email,"
                "banned"
                ")"
            ")"
        ),
        "$skip": num_issues_to_skip,
        "$top": num_issues_to_retrieve
    }

# Request Issues from the API
print(f"Requesting {num_issues_to_retrieve} issues from YouTrack starting with Id {num_issues_to_skip+1}...")
issues_list = requests.get(api_url+"admin/projects/"+project_id+"/issues", params=issues_list_fields, headers=auth_header)
all_issues = issues_list.json()



Requesting 10 from YouTrack starting with Id 600...


In [5]:
# unpack issues from youtrack's json output format to a simpler json format that can be normalized to csv
print("Unpacking Issues...")
all_unpacked_issues = [unpackers.unpack_youtrack_issue(issue) for issue in all_issues]

# flatten out json. This still leaves some columns as lists/series
print("Flattening Issues...")
flattened_unpacked_issues = pd.json_normalize(all_unpacked_issues)

# expand list/series values into multiple columns of the same name. (This matches jira's import syntax requirements for mult-value fields)
print("Expanding Issues...")
df_list = [
    flattened_unpacked_issues[col].apply(unpackers.flatten_series_to_columns, args=[col]) 
    for col in flattened_unpacked_issues
    ]

# smoosh dataframes from expanded columns together to form final csv
flattened_unpacked_expanded_issues = pd.concat(df_list, axis=1)

print("Unmangling header row...")
# write csv. Pandas doesnt allow non unique column names so we will have to do some additional processing on the csv after this
intermediate_csv_path = './data/flattened_unpacked_expanded_issues.csv'
flattened_unpacked_expanded_issues.to_csv(intermediate_csv_path)

# Unmangle the non-unique column names, write a clean header to a new csv and copy all of the data.
with open('./data/flattened_unpacked_expanded_issues.csv','r') as old_file, \
    open('./data/final_jira_issues.csv', 'w') as new_file:
    reader = csv.reader(old_file, delimiter=',')
    writer = csv.writer(new_file, delimiter=',')
    old_header = next(reader)
    new_header = [col.split(":")[0] for col in old_header]
    writer.writerow(new_header)
    [writer.writerow(line) for line in reader]

# delete intermediate csv
print("Cleaning up...")
os.remove(intermediate_csv_path)
final_csv_path = './data/final_jira_issues.csv'

print(Fore.GREEN + Style.BRIGHT + f"YouTrack Export Complete. See {final_csv_path}" + Fore.RESET + Style.RESET_ALL)

Unpacking Issues...
Flattening Issues...
Expanding Issues...
Unmangling header row...
Cleaning up...
YouTrack Export Complete. See ./data/final_jira_issues.csv
